# आव्हान: डेटा सायन्सबद्दल मजकूराचे विश्लेषण करणे

या उदाहरणात, चला पारंपारिक डेटा सायन्स प्रक्रियेच्या सर्व टप्प्यांचा समावेश असलेले एक साधे व्यायाम करूया. तुम्हाला कोणताही कोड लिहण्याची गरज नाही, तुम्ही फक्त खालील सेलवर क्लिक करून ते चालवू शकता आणि परिणाम पाहू शकता. आव्हानस्वरूप, तुम्हाला वेगवेगळ्या डेटासोबत हा कोड वापरून पहाण्यास प्रोत्साहित केले आहे.

## लक्ष्य

या धड्यात, आपण डेटा सायन्सशी संबंधित वेगवेगळ्या संकल्पनांवर चर्चा केली आहे. चला काही **मजकूर मायनिंग** करून आणखी संबंधित संकल्पना शोधण्याचा प्रयत्न करूया. आपण डेटा सायन्सबद्दलचा एक मजकूर घेऊ, त्यातून कीवर्ड्स काढू, आणि नंतर परिणाम व्हिज्युअलाईज करण्याचा प्रयत्न करू.

मजकूर म्हणून, मी विकिपीडियावर डेटा सायन्सविषयी पृष्ठ वापरणार आहे:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## पायरी 1: डेटा मिळविणे

प्रत्येक डेटा सायन्स प्रक्रियेतील पहिली पायरी म्हणजे डेटा मिळविणे. हे करण्यासाठी आपण `requests` लायब्ररी वापरणार आहोत:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## पाऊल 2: डेटा रूपांतरित करणे

पुढील पाऊल म्हणजे डेटा प्रक्रिया करण्यासाठी योग्य स्वरूपात रुपांतरित करणे. आमच्या बाबतीत, आम्ही पृष्ठावरून HTML स्रोत कोड डाउनलोड केला आहे, आणि आम्हाला ते साध्या मजकूरात रूपांतरित करणे आवश्यक आहे.

हे करण्यास अनेक मार्ग आहेत. आम्ही [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), HTML पार्सिंगसाठी एक लोकप्रिय Python लायब्ररी वापरणार आहोत. BeautifulSoup आपल्याला विशिष्ट HTML घटकांवर लक्ष केंद्रित करण्याची परवानगी देते, त्यामुळे आपण विकिपीडिया मधील मुख्य लेखातील सामग्रीवर लक्ष केंद्रित करू शकतो व काही नेव्हिगेशन मेनू, साइडबार, फूटर्स आणि इतर अप्रासंगिक सामग्री कमी करू शकतो (जरी काही बोइलरप्लेट मजकूर अजूनही उरू शकतो).


प्रथम, आपल्याला HTML पार्सिंगसाठी BeautifulSoup लायब्ररी इन्स्टॉल करणे आवश्यक आहे:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## टप्पा 3: अंतर्दृष्टी प्राप्त करणे

सर्वात महत्त्वाचा टप्पा म्हणजे आपला डेटा अशा स्वरूपात रूपांतरित करणे ज्यातून आपण अंतर्दृष्टी काढू शकू. आमच्या प्रकरणात, आम्हाला मजकूरातून कीवर्ड काढायचे आहेत, आणि कोणते कीवर्ड अधिक अर्थपूर्ण आहेत हे पाहायचे आहे.

आम्ही कीवर्ड काढण्यासाठी Python लायब्ररी [RAKE](https://github.com/aneesha/RAKE) वापरणार आहोत. प्रथम, ही लायब्ररी अस्तित्वात नसेल तर ती इन्स्टॉल करूया: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

मुख्य कार्यक्षमता `Rake` ऑब्जेक्टमधून उपलब्ध आहे, ज्याला आपण काही पॅरामीटर्स वापरून सानुकूलित करू शकतो. आपल्या प्रकरणात, आपण कीवर्डची किमान लांबी 5 अक्षरे, कागदपत्रातील कीवर्डची किमान वारंवारिता 3, आणि कीवर्डमधील शब्दांची कमाल संख्या 2 ठरवणार आहोत. इतर मूल्यांसह खेळायला मोकळे व्हा आणि परिणामाचे निरीक्षण करा.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


आम्हाला संबंधित महत्त्वाचा दर्जा असलेल्या अटींची यादी प्राप्त झाली. आपण पाहू शकता की, सर्वाधिक संबंधित विषयजागा, जसे की मशीन लर्निंग आणि बिग डेटा, यादीतील वरच्या स्थानांवर आहेत.

## टप्पा 4: निकालाचे दृश्यीकरण

लोक डेटा दृश्य रूपात सर्वोत्तम समजून घेतात. म्हणूनच काही माहिती मिळवण्यासाठी डेटा दृश्य स्वरूपात सादर करणे अनेकदा अर्थपूर्ण असते. आम्ही Python मध्ये `matplotlib` लायब्ररी वापरून कीवर्ड्सच्या संबंधिततेसह सोपी वितरण रेखाटू शकतो:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

तथापि, शब्द वारंवारता दर्शविण्याचा आणखी उत्कृष्ट मार्ग आहे - **शब्द मेघ** वापरणे. आपल्याला आपल्या कीवर्ड यादीमधून शब्द मेघ काढण्यासाठी आणखी एक लायब्ररी स्थापित करावी लागेल.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` ऑब्जेक्ट मूळ मजकूर, किंवा आधी मोजलेल्या शब्दांची यादी त्यांच्या वारंवारतेसह, स्वीकारण्याची आणि नंतर `matplotlib` वापरून दर्शविली जाऊ शकणारी प्रतिमा परत करण्याची जबाबदारी घेतो:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

आपण मूळ मजकूरसुद्धा `WordCloud` मध्ये देऊ शकतो - पाहू या की आपल्याला समान परिणाम मिळतो का:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

तुम्हाला आत्ताच दिसेल की शब्द बादल आता अधिक प्रभावशाली दिसते, परंतु त्यात खूप आवाज देखील आहे (उदा. `Retrieved on` सारखे असंबंधित शब्द). तसेच, आपल्याला दोन शब्दांत बनलेल्या कमी कीवर्ड्स मिळतात, जसे *data scientist*, किंवा *computer science*. कारण RAKE अल्गोरिदम मजकुरातून चांगल्या कीवर्ड्स निवडण्यात खूप चांगले कार्य करते. ही उदाहरण डेटा प्री-प्रोसेसिंग आणि क्लिनिंगच्या महत्त्वाचे महत्त्व दर्शवते, कारण शेवटी स्पष्ट चित्र आपल्याला चांगले निर्णय घेण्यास अनुमती देईल.

या व्यायामात आपण विकिपीडिया मजकुरातून कीवर्ड्स आणि शब्द बादलाच्या स्वरूपात काही अर्थ काढण्याच्या साध्या प्रक्रियेतून गेलो आहोत. हे उदाहरण अतिशय सोपे आहे, पण हे प्रदर्शित करते की डेटा सायंटिष्ट जेव्हा डेटाबरोबर काम करतो तेव्हा तो सर्वसाधारण टप्पे कोणते असतात, डेटा मिळवण्यापासून सुरू होऊन दृश्यासाठी.

आपल्या अभ्यासक्रमात आपण या सर्व टप्प्यांवर तपशीलवार चर्चा करू.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
